# The complete test pipeline: both endings of your vision

Every run now ends in exactly one of the two outcomes you designed:

```
verify -> train[:candidate] -> LIGHT gate
  |
  +-- [PASS] -> register -> HEAVY EVAL (artifact check + HTML report)
  |               +-- [PASS] -> promote model (alias) + promote image (retag)
  |               +-- [FAIL] -> version stays in Registry, unaliased (diagnosable)
  |
  +-- [FAIL] -> REJECTION REPORT (HTML): why the model was too weak
  |
  (log to Experiments: always, both endings)
```

**New: `rejection_report_component` in `dsl.Else`.** Until now a light-gate
rejection was silence. Now it produces an HTML report with two **gauge
visualizations** - a scale with the threshold line and a dot showing where the
model landed (red zone = the reason for rejection):
- gauge 1: lower 95% CI bound vs `accuracy_threshold` (dot must be RIGHT of the line)
- gauge 2: std/spread vs `max_std_threshold` (dot must be LEFT of the line)

This makes the rejection *legible*: the mean can look fine while the lower-CI
bound fails - the gauges show exactly which criterion failed and by how much.
Lightweight component (no fsl needed - it only has numbers), pure-SVG, no deps.

**Verified offline (KFP 2.16.1 + GCPC 2.22.0):** `dsl.Else` compiles; the
compiled graph shows `condition-branches-1` with the success branch
(register -> heavy -> nested promotion) and the failure branch
(rejection-report); gauge SVG logic unit-tested in both directions
(right-pass for accuracy, left-pass for std). NOT verified: the run itself.

**No image rebuild needed** - the new component is lightweight.

**How to SEE the rejection branch fire** (recommended once): set
`accuracy_threshold: 0.99` in the config, run - the light gate fails, the graph
takes the else-branch, and the rejection report renders under the node's
artifacts. Restore 0.90 afterwards. Same both-cases discipline as always.

## Setup

In [1]:
# %pip install kfp google-cloud-pipeline-components
# (restart kernel after installing)

## Load config

In [2]:
import json

with open("configs/pipeline_config.json") as f:
    CFG = json.load(f)

PROJECT = CFG["project"]
REGION  = CFG["region"]
BUCKET  = CFG["bucket"]
DATASET = CFG["dataset"]
REPO    = CFG["pipeline_template_repo"]
PIPELINE_YAML = CFG["pipeline_yaml_path"]
IMAGE   = CFG["training_image_uri"]

for name, val in [("project", PROJECT), ("bucket", BUCKET),
                  ("training image", IMAGE),
                  ("serving image", CFG["serving_container_image_uri"])]:
    assert "YOUR" not in val and "prediction" not in val, f"config field '{name}' needs fixing: {val}"
    print(f"{name}: {val}")
print("gate:", CFG["evaluation"])

project: dark-data-discovery
bucket: dark-data-discovery-fsl-data
training image: us-central1-docker.pkg.dev/dark-data-discovery/fsl-images/train:candidate
serving image: us-central1-docker.pkg.dev/dark-data-discovery/fsl-images/train:candidate
gate: {'_comment': "Gate criteria. accuracy_threshold: the lower bound of the 95% CI (mean - ci95) must be >= this, so we're statistically confident the model is above the bar, not just on average. max_std_threshold: the model must also be stable - std across test episodes <= this. Add more criteria here (e.g. regression-vs-previous) and wire them into the gate's checks list.", 'accuracy_threshold': 0.99, 'max_std_threshold': 0.05}


## Component 1: `verify_frozen_component`

In [3]:
from typing import NamedTuple
from kfp import dsl
from kfp.dsl import ContainerSpec, OutputPath, Output, Metrics, HTML


@dsl.component(base_image="python:3.10", packages_to_install=["google-cloud-storage"])
def verify_frozen_component(project: str, bucket: str, dataset: str) -> str:
    from google.cloud import storage
    import json as _json
    c = storage.Client(project=project)
    m = _json.loads(c.bucket(bucket).blob(f"raw/{dataset}/MANIFEST.json").download_as_bytes().decode("utf-8"))
    return f"VERIFIED sha={m['archive_sha256'][:12]}"

## Component 2: `train_container`

In [4]:
IMAGE = CFG['training_image_uri']


@dsl.container_component
def train_container(
    project: str,
    bucket: str,
    dataset: str,
    n_way: int,
    k_shot: int,
    query: int,
    seed: int,
    train_iters: int,
    embedding_hid: int,
    accuracy_mean: OutputPath(float),
    accuracy_ci95: OutputPath(float),
    accuracy_std: OutputPath(float),
    train_time: OutputPath(float),
    model_dir: OutputPath(str),
    data_sha: OutputPath(str),
):
    return ContainerSpec(
        image=IMAGE,
        command=["python", "scripts/train_pipeline_entry.py"],
        args=[
            "--project", project, "--bucket", bucket, "--dataset", dataset,
            "--n-way", n_way, "--k-shot", k_shot, "--query", query,
            "--seed", seed, "--train-iters", train_iters, "--embedding-hid", embedding_hid,
            "--accuracy-mean-output-path", accuracy_mean,
            "--accuracy-ci95-output-path", accuracy_ci95,
            "--accuracy-std-output-path", accuracy_std,
            "--train-time-output-path", train_time,
            "--model-dir-output-path", model_dir,
            "--data-sha-output-path", data_sha,
        ],
    )

## Component 3: `heavy_eval_container`

In [5]:
# --- CIEZKA EWALUACJA (kontener): laduje ZAPISANY artefakt, przelicza niezaleznie ---
@dsl.container_component
def heavy_eval_container(
    project: str,
    region: str,
    bucket: str,
    dataset: str,
    model_dir: str,
    model_version: str,
    train_accuracy_mean: float,
    train_accuracy_ci95: float,
    accuracy_threshold: float,
    max_std_threshold: float,
    test_episodes: int,
    passed: OutputPath(bool),
    reason: OutputPath(str),
    accuracy_recomputed: OutputPath(float),
    report: Output[HTML],
):
    return ContainerSpec(
        image=IMAGE,
        command=["python", "scripts/evaluate_pipeline_entry.py"],
        args=[
            "--project", project, "--region", region, "--bucket", bucket,
            "--dataset", dataset, "--model-dir", model_dir,
            "--model-version", model_version,
            "--train-accuracy-mean", train_accuracy_mean,
            "--train-accuracy-ci95", train_accuracy_ci95,
            "--accuracy-threshold", accuracy_threshold,
            "--max-std-threshold", max_std_threshold,
            "--test-episodes", test_episodes,
            "--passed-output-path", passed,
            "--reason-output-path", reason,
            "--accuracy-recomputed-output-path", accuracy_recomputed,
            "--report-html-path", report.path,
        ],
    )

## Wrap training into a Custom Job

In [6]:
from google_cloud_pipeline_components.v1.custom_job import create_custom_training_job_op_from_component

# Trening = Custom Job (dedykowana maszyna, widoczny w Vertex Training),
# ktorego workerem jest NASZ kontener z fsl. Kontener + job naraz.
train_as_job = create_custom_training_job_op_from_component(
    train_container,
    display_name="fsl-training-job",
    machine_type=CFG["compute"]["machine_type"],
    replica_count=1,
)

## Component 4: `evaluate_gate_component` (light gate)

In [7]:
@dsl.component(base_image="python:3.10")
def evaluate_gate_component(
    accuracy_mean: float, accuracy_ci95: float, accuracy_std: float,
    accuracy_threshold: float, max_std_threshold: float,
    gate_metrics: Output[Metrics],
) -> NamedTuple("G", [("passed", bool), ("reason", str)]):
    checks = []
    lower = accuracy_mean - accuracy_ci95
    checks.append(("accuracy_lower_ci", lower >= accuracy_threshold,
                   f"lowerCI={lower:.4f} vs threshold={accuracy_threshold:.4f}"))
    checks.append(("stability", accuracy_std <= max_std_threshold,
                   f"std={accuracy_std:.4f} vs max={max_std_threshold:.4f}"))
    passed = all(o for _, o, _ in checks)
    reason = "; ".join(f"{n}: {'PASS' if o else 'FAIL'} ({d})" for n, o, d in checks)
    gate_metrics.log_metric("gate_passed", 1.0 if passed else 0.0)
    gate_metrics.log_metric("accuracy_lower_ci", lower)
    gate_metrics.log_metric("margin_above_threshold", lower - accuracy_threshold)
    gate_metrics.log_metric("std_headroom", max_std_threshold - accuracy_std)
    print(f"Gate: {'PASSED' if passed else 'FAILED'} - {reason}")
    from collections import namedtuple
    return namedtuple("G", ["passed", "reason"])(passed, reason)

## Component 5: `register_model_component`

In [8]:
@dsl.component(base_image="python:3.10", packages_to_install=["google-cloud-aiplatform"])
def register_model_component(
    project: str, region: str, model_display_name: str,
    serving_container_image_uri: str, model_dir: str,
    accuracy_mean: float, data_sha: str, seed: int,
) -> NamedTuple("Reg", [("resource_name", str), ("version_id", str)]):
    from google.cloud import aiplatform
    aiplatform.init(project=project, location=region)
    existing = aiplatform.Model.list(filter=f'display_name="{model_display_name}"')
    parent = existing[0].resource_name if existing else None
    m = aiplatform.Model.upload(
        display_name=model_display_name, artifact_uri=model_dir,
        serving_container_image_uri=serving_container_image_uri, parent_model=parent,
        labels={"framework": "pytorch", "model": "protonet", "seed": str(seed)},
        description=f"ProtoNet acc={accuracy_mean:.4f} sha={data_sha[:12]}",
    )
    print(f"Registered {m.resource_name} v{m.version_id}")
    from collections import namedtuple
    return namedtuple("Reg", ["resource_name", "version_id"])(m.resource_name, str(m.version_id))

## Component 6: `log_experiment_component` (unconditional)

In [9]:
@dsl.component(base_image="python:3.10", packages_to_install=["google-cloud-aiplatform"])
def log_experiment_component(
    project: str, region: str, experiment_name: str, run_type: str, dataset: str,
    n_way: int, k_shot: int, query: int, seed: int, train_iters: int,
    data_sha: str, accuracy_mean: float, accuracy_ci95: float,
    accuracy_std: float, train_time: float, gate_passed: bool, gate_reason: str,
) -> str:
    import time
    from google.cloud import aiplatform
    aiplatform.init(project=project, location=region, experiment=experiment_name)
    run_name = f"pipeline-{run_type.replace('_','-')}-{n_way}w{k_shot}s-seed{seed}-{int(time.time())}"
    aiplatform.start_run(run_name)
    aiplatform.log_params({"model": "protonet", "n_way": n_way, "k_shot": k_shot,
        "query": query, "seed": seed, "train_iters": train_iters, "dataset": dataset,
        "dataset_archive_sha256": data_sha, "run_type": run_type,
        "gate_passed": gate_passed, "gate_reason": gate_reason})
    aiplatform.log_metrics({"test_accuracy_mean": accuracy_mean,
        "test_accuracy_ci95": accuracy_ci95, "test_accuracy_std": accuracy_std,
        "train_time_seconds": train_time})
    aiplatform.end_run()
    print(f"Logged run {run_name}")
    return run_name

## Component 7: `rejection_report_component` (NEW - the failure ending)

Lightweight. Renders the two gauges + verdict + the gate's reason string.

In [10]:
# --- RAPORT ODRZUCENIA (lekki): HTML wyjasniajacy, dlaczego bramka nie puscila ---
@dsl.component(base_image="python:3.10")
def rejection_report_component(
    accuracy_mean: float, accuracy_ci95: float, accuracy_std: float,
    accuracy_threshold: float, max_std_threshold: float, gate_reason: str,
    report: Output[HTML],
) -> str:
    def svg_gauge(value, threshold, lo, hi, label, value_label, threshold_label,
                  width=640, height=90, pass_side="right"):
        pad = 60
        bar_w = width - 2 * pad
        y = 42

        def x_px(v):
            v = max(lo, min(hi, v))
            return pad + (v - lo) / (hi - lo or 1) * bar_w

        tx, vx = x_px(threshold), x_px(value)
        if pass_side == "right":
            fail_rect = f'<rect x="{pad}" y="{y}" width="{tx-pad:.1f}" height="14" fill="#f6d5d1"/>'
            pass_rect = f'<rect x="{tx:.1f}" y="{y}" width="{pad+bar_w-tx:.1f}" height="14" fill="#d9ead9"/>'
        else:
            pass_rect = f'<rect x="{pad}" y="{y}" width="{tx-pad:.1f}" height="14" fill="#d9ead9"/>'
            fail_rect = f'<rect x="{tx:.1f}" y="{y}" width="{pad+bar_w-tx:.1f}" height="14" fill="#f6d5d1"/>'
        ok = (value >= threshold) if pass_side == "right" else (value <= threshold)
        col = "#2a7d4f" if ok else "#c0392b"
        return (f'<svg viewBox="0 0 {width} {height}" xmlns="http://www.w3.org/2000/svg">'
                f'<text x="{pad}" y="20" font-size="13" fill="#333" font-weight="600">{label}</text>'
                f'{fail_rect}{pass_rect}'
                f'<line x1="{tx:.1f}" y1="{y-6}" x2="{tx:.1f}" y2="{y+20}" stroke="#c0392b" stroke-width="2"/>'
                f'<text x="{tx:.1f}" y="{y+34}" font-size="11" fill="#c0392b" text-anchor="middle">{threshold_label}</text>'
                f'<circle cx="{vx:.1f}" cy="{y+7}" r="7" fill="{col}"/>'
                f'<text x="{vx:.1f}" y="{y-10}" font-size="11" fill="{col}" text-anchor="middle" font-weight="600">{value_label}</text>'
                f'</svg>')

    lower = accuracy_mean - accuracy_ci95
    g1 = svg_gauge(lower, accuracy_threshold, min(0.5, lower - 0.05), 1.0,
                   "Criterion 1: lower 95% CI bound (must be RIGHT of the threshold)",
                   f"{lower:.3f}", f"threshold {accuracy_threshold}", pass_side="right")
    g2 = svg_gauge(accuracy_std, max_std_threshold, 0.0, max(0.15, accuracy_std * 1.5),
                   "Criterion 2: std / spread (must be LEFT of the max)",
                   f"{accuracy_std:.3f}", f"max {max_std_threshold}", pass_side="left")
    html = f"""<!DOCTYPE html><html><head><style>
body{{font-family:-apple-system,sans-serif;margin:24px;color:#1a1a1a;max-width:760px}}
.verdict{{font-size:18px;font-weight:700;color:#c0392b;margin:10px 0}}
.metric{{display:inline-block;margin:8px 28px 8px 0}}.metric .v{{font-size:24px;font-weight:600;color:#4C72B0}}
.metric .l{{font-size:12px;color:#666}}.reason{{font-size:13px;color:#555;background:#f7f7f7;padding:10px;border-radius:6px}}
.meta{{font-size:12px;color:#888}}</style></head><body>
<h2>Gate Rejection Report</h2>
<div class="verdict">MODEL REJECTED - not registered, not promoted</div>
<div class="metric"><div class="v">{accuracy_mean:.2%}</div><div class="l">mean accuracy</div></div>
<div class="metric"><div class="v">{lower:.4f}</div><div class="l">lower 95% CI</div></div>
<div class="metric"><div class="v">{accuracy_std:.4f}</div><div class="l">std (spread)</div></div>
{g1}{g2}
<p class="reason">{gate_reason}</p>
<p class="meta">A dot in the red zone is the reason for rejection. The mean alone can look fine while
the lower-CI bound (confidence, penalized by spread) or the spread itself fails - see the two gauges.</p>
</body></html>"""
    with open(report.path, "w") as f:
        f.write(html)
    print(f"Rejection report written. Reason: {gate_reason}")
    return "rejected-report-written"

## Components 8+9: promotion (model alias + image retag)

In [11]:
# --- PROMOCJA (lekkie): alias "production" na wersji + retag obrazu ---
@dsl.component(base_image="python:3.10", packages_to_install=["google-cloud-aiplatform"])
def promote_model_component(
    project: str, region: str, model_resource_name: str, version_id: str,
) -> str:
    from google.cloud import aiplatform
    from google.cloud.aiplatform.models import ModelRegistry
    aiplatform.init(project=project, location=region)
    registry = ModelRegistry(model=model_resource_name, project=project, location=region)
    registry.add_version_aliases(["production"], version=version_id)
    print(f"Alias 'production' -> version {version_id} of {model_resource_name}")
    return f"{model_resource_name}@{version_id}"


@dsl.component(base_image="python:3.10", packages_to_install=["google-cloud-artifact-registry"])
def promote_image_component(
    project: str, region: str, repo: str, image_name: str,
    source_tag: str, target_tag: str,
) -> str:
    # Retag bez pobierania obrazu: tag docelowy wskazuje na TE SAMA wersje
    # (digest), na ktora wskazuje tag zrodlowy. Testowany artefakt == promowany.
    from google.api_core.exceptions import AlreadyExists, NotFound
    from google.cloud import artifactregistry_v1
    client = artifactregistry_v1.ArtifactRegistryClient()
    base = f"projects/{project}/locations/{region}/repositories/{repo}/packages/{image_name}"
    src = client.get_tag(name=f"{base}/tags/{source_tag}")
    print(f"Source {source_tag} -> version: {src.version}")
    tag_obj = artifactregistry_v1.Tag(name=f"{base}/tags/{target_tag}", version=src.version)
    try:
        client.create_tag(parent=base, tag=tag_obj, tag_id=target_tag)
        print(f"Created tag {target_tag}")
    except AlreadyExists:
        client.update_tag(tag=tag_obj)
        print(f"Updated tag {target_tag} to {src.version}")
    return f"{image_name}:{target_tag} -> {src.version.split('/')[-1][:24]}"

## Pipeline: both endings

In [12]:
@dsl.pipeline(name="fsl-hello-pipeline",
              description="verify -> train[CustomJob+container] -> gate -> If(register)")
def training_pipeline(
    project: str, bucket: str, dataset: str = "omniglot", region: str = "us-central1",
    n_way: int = 5, k_shot: int = 5, query: int = 15, seed: int = 0,
    train_iters: int = 300, embedding_hid: int = 64, test_episodes: int = 1000,
    model_display_name: str = "fsl-protonet-omniglot",
    serving_container_image_uri: str = "us-central1-docker.pkg.dev/x/fsl-images/train:candidate",
    accuracy_threshold: float = 0.90, max_std_threshold: float = 0.05,
    image_repo: str = "fsl-images", image_name: str = "train",
    run_type: str = "inner_loop", experiment_name: str = "fsl-inner-loop",
):
    verify_task = verify_frozen_component(project=project, bucket=bucket, dataset=dataset)

    train_task = train_as_job(
        project=project, location=region,
        bucket=bucket, dataset=dataset, n_way=n_way, k_shot=k_shot, query=query,
        seed=seed, train_iters=train_iters, embedding_hid=embedding_hid,
    ).after(verify_task)

    gate_task = evaluate_gate_component(
        accuracy_mean=train_task.outputs["accuracy_mean"],
        accuracy_ci95=train_task.outputs["accuracy_ci95"],
        accuracy_std=train_task.outputs["accuracy_std"],
        accuracy_threshold=accuracy_threshold, max_std_threshold=max_std_threshold,
    )

    with dsl.If(gate_task.outputs["passed"] == True, name="gate-passed"):
        register_task = register_model_component(
            project=project, region=region, model_display_name=model_display_name,
            serving_container_image_uri=serving_container_image_uri,
            model_dir=train_task.outputs["model_dir"],
            accuracy_mean=train_task.outputs["accuracy_mean"],
            data_sha=train_task.outputs["data_sha"], seed=seed,
        )

        # CIEZKA EWALUACJA zarejestrowanej wersji: laduje artefakt z GCS,
        # przelicza niezaleznie, testuje spojnosc, renderuje raport HTML.
        heavy_task = heavy_eval_container(
            project=project, region=region, bucket=bucket, dataset=dataset,
            model_dir=train_task.outputs["model_dir"],
            model_version=register_task.outputs["version_id"],
            train_accuracy_mean=train_task.outputs["accuracy_mean"],
            train_accuracy_ci95=train_task.outputs["accuracy_ci95"],
            accuracy_threshold=accuracy_threshold,
            max_std_threshold=max_std_threshold,
            test_episodes=test_episodes,
        )

        # DRUGA BRAMKA: promocja tylko gdy zapisany artefakt zweryfikowany
        with dsl.If(heavy_task.outputs["passed"] == True, name="heavy-passed"):
            promote_model_component(
                project=project, region=region,
                model_resource_name=register_task.outputs["resource_name"],
                version_id=register_task.outputs["version_id"],
            )
            promote_image_component(
                project=project, region=region,
                repo=image_repo, image_name=image_name,
                source_tag="candidate", target_tag="production",
            )

    # LOGOWANIE: bezwarunkowe (poza If) - loguje kazdy przebieg, tez odrzucony
    with dsl.Else(name="gate-failed"):
        rejection_report_component(
            accuracy_mean=train_task.outputs["accuracy_mean"],
            accuracy_ci95=train_task.outputs["accuracy_ci95"],
            accuracy_std=train_task.outputs["accuracy_std"],
            accuracy_threshold=accuracy_threshold,
            max_std_threshold=max_std_threshold,
            gate_reason=gate_task.outputs["reason"],
        )

    log_experiment_component(
        project=project, region=region, experiment_name=experiment_name, run_type=run_type,
        dataset=dataset, n_way=n_way, k_shot=k_shot, query=query, seed=seed, train_iters=train_iters,
        data_sha=train_task.outputs["data_sha"],
        accuracy_mean=train_task.outputs["accuracy_mean"],
        accuracy_ci95=train_task.outputs["accuracy_ci95"],
        accuracy_std=train_task.outputs["accuracy_std"],
        train_time=train_task.outputs["train_time"],
        gate_passed=gate_task.outputs["passed"],
        gate_reason=gate_task.outputs["reason"],
    )

## Compile and submit

In [13]:
from kfp import compiler
from google.cloud import aiplatform

compiler.Compiler().compile(training_pipeline, PIPELINE_YAML)
print("Compiled ->", PIPELINE_YAML)

aiplatform.init(project=PROJECT, location=REGION)
parameter_values = {
    "project": PROJECT, "bucket": BUCKET, "dataset": DATASET, "region": REGION,
    "model_display_name": CFG["model_display_name"],
    "serving_container_image_uri": CFG["serving_container_image_uri"],
    "accuracy_threshold": CFG["evaluation"]["accuracy_threshold"],
    "max_std_threshold": CFG["evaluation"]["max_std_threshold"],
    "run_type": CFG["run_type"], "experiment_name": CFG["experiment_name"],
    "image_repo": "fsl-images", "image_name": "train",
    **{k: CFG["training"][k] for k in ["n_way","k_shot","query","seed","train_iters","embedding_hid","test_episodes"]},
}
job = aiplatform.PipelineJob(
    display_name="fsl-test-pipeline-complete",
    template_path=PIPELINE_YAML,
    pipeline_root=f"gs://{BUCKET}/pipeline-root",
    parameter_values=parameter_values,
)
job.submit()
print("Submitted. At threshold 0.90 expect the success branch (as before).")
print("To see the rejection report: set accuracy_threshold 0.99, resubmit,")
print("click rejection-report-component -> artifacts -> report.")

/home/jupyter/envs/fsl/lib/python3.10/site-packages/google/api_core/_python_version_support.py:255: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.cloud.aiplatform_v1 once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.cloud.aiplatform_v1 past that date.
  warnings.warn(message, FutureWarning)
/home/jupyter/envs/fsl/lib/python3.10/site-packages/google/api_core/_python_version_support.py:255: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.cloud.aiplatform_v1beta1 once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.cloud.aiplatform_v1beta1 past that date.
  warnings.warn(message, FutureWarning)
/home/jupyter/envs/fsl/lib/python3.10/site-packages/

Compiled -> train_pipeline.yaml
Creating PipelineJob
PipelineJob created. Resource name: projects/815755318672/locations/us-central1/pipelineJobs/fsl-hello-pipeline-20260708225712
To use this PipelineJob in another session:
pipeline_job = aiplatform.PipelineJob.get('projects/815755318672/locations/us-central1/pipelineJobs/fsl-hello-pipeline-20260708225712')
View Pipeline Job:
https://console.cloud.google.com/vertex-ai/locations/us-central1/pipelines/runs/fsl-hello-pipeline-20260708225712?project=815755318672
Submitted. At threshold 0.90 expect the success branch (as before).
To see the rejection report: set accuracy_threshold 0.99, resubmit,
click rejection-report-component -> artifacts -> report.


## Publish template (auto-incremented)

In [14]:
import sys
sys.path.insert(0, "scripts")
from publish_template import publish_next_version

publish_next_version(project=PROJECT, region=REGION, repo=REPO,
    yaml_path=PIPELINE_YAML,
    description="Complete test pipeline: both endings (promotion / rejection report)")

Existing version tags: ['latest', 'v1', 'v10', 'v11', 'v2', 'v3', 'v4', 'v5', 'v6', 'v7', 'v8', 'v9'] -> assigning v12
Published fsl-hello-pipeline @ v12 (version id: sha256:fa86a4d8d00b780f73d5ae859be0a3fce849661b86ce45be35a082faeeeaff27)


('fsl-hello-pipeline',
 'sha256:fa86a4d8d00b780f73d5ae859be0a3fce849661b86ce45be35a082faeeeaff27',
 'v12')

## What's left from the vision

- **Explainability container** (prototype distances, nearest support examples,
  PCA-SVG embedding map) - parallel to promotion in the success branch, always-on
- **Production pipeline** - same components, `:production` image, weekly
  trigger, alias-move as deploy